# Generative AI as a Task Shock## Full Empirical Pipeline Notebook — v2**Author:** Hakan Zeki Gülmez · TUM School of Management  **Supervisor:** Prof. Dr. Helmut Farbmacher  Pipeline: `build_firm_universe.py` → `collect_10k_text.py` → `score_literature_rubric.py` → `build_financial_panel.py` → `did_main.R`

## Section 0: Setup & Data Loading

In [ ]:
import pandas as pdimport numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltimport matplotlib.ticker as mtickerimport warningswarnings.filterwarnings('ignore')plt.rcParams.update({    'font.size': 11, 'axes.titlesize': 13, 'axes.labelsize': 12,    'figure.dpi': 150, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})# Load datascores = pd.read_csv('data/processed/lit_scores.csv')panel = pd.read_csv('data/processed/financial_panel.csv')universe = pd.read_csv('data/raw/firm_universe.csv')panel['period_end'] = pd.to_datetime(panel['period_end'])# Merge scores + paneldf = panel.merge(scores[['ticker','normalized_score','raw_score',    'C1','C2','C3','C4','C5','C6','C7','C8','C9','C10']], on='ticker')df['post'] = (df['period_end'] >= '2022-10-01').astype(int)df['ln_revenue'] = np.log(df['revenue'].replace(0, np.nan))df['rho'] = df['normalized_score']# Pre-shock statspre = df[df['post']==0]pre_med = pre.groupby('ticker')['revenue'].median()pre_q = pre.groupby('ticker').size()df['pre_rev'] = df['ticker'].map(pre_med)df['pre_rev_m'] = df['pre_rev'] / 1e6df['pre_q'] = df['ticker'].map(pre_q)# Exclude non-B2Bexclude = [    'RBLX','PLTK','SKLZ','TTWO','EA','MSGM','MYPS','MYSZ',    'BMBL','MTCH','CARG','CARS','TRIP','GDRX','ZDGE','IAC',    'DUOL','COUR','NRDS','META','SNAP','PINS','GOOGL','NXDR',    'IONQ','RGTI','DDD','AUUD','ALBT','CEVA','ISSC','PXLW',    'AUR','PDYN','RCAT','XTIA','ODYS','TWST','GGRP',    'AIFF','PHUN','RIME','FNGR','NXTT','IPM','INOD','RDNW',    'WFCF','HCTI','YYAI','AUID','TWAV','SLE','VRME',    'CTSH','DXC','SAIC','PSN','CACI',]df = df[~df['ticker'].isin(exclude)]# Sample flagsdf['sme_200'] = ((df['pre_rev_m'] < 200) & (df['pre_rev_m'] > 5) & (df['pre_q'] >= 6)).astype(int)df['sme_500'] = ((df['pre_rev_m'] < 500) & (df['pre_rev_m'] > 5) & (df['pre_q'] >= 6)).astype(int)sme = df[df['sme_200']==1].copy()print(f'Universe: {universe.shape[0]} firms')print(f'Scored: {scores.shape[0]} firms')print(f'B2B SaaS panel: {df["ticker"].nunique()} firms')print(f'SME <$200M sample: {sme["ticker"].nunique()} firms')print(f'SME <$500M sample: {df[df["sme_500"]==1]["ticker"].nunique()} firms')

## Section 1: Sample Construction Funnel → Fig 7

In [ ]:
# Figure 7: Sample funnelfig, ax = plt.subplots(figsize=(10, 7))ax.set_xlim(0, 10)ax.set_ylim(0, 10)ax.axis('off')steps = [    (f'SEC EDGAR SIC 7370-7379\n{universe.shape[0]} firms', 0.95),    (f'NYSE/Nasdaq + XBRL coverage\n{universe.shape[0]} firms', 0.80),    (f'10-K Item 1 extracted\n{scores.shape[0]} firms', 0.65),    (f'B2B SaaS filter (excl gaming/hardware/consumer)\n{df["ticker"].nunique()} firms', 0.50),    (f'SME <$200M, >$5M, >=6 pre-shock quarters\n{sme["ticker"].nunique()} firms', 0.35),]for i, (label, y) in enumerate(steps):    width = 0.9 - i * 0.08    rect = plt.Rectangle((5 - width*5, y*10 - 0.6), width*10, 1.0,                          facecolor=plt.cm.Blues(0.3 + i*0.15), edgecolor='black', lw=1.5)    ax.add_patch(rect)    ax.text(5, y*10 - 0.1, label, ha='center', va='center', fontsize=11, fontweight='bold')ax.set_title('Sample Construction Funnel', fontsize=14, fontweight='bold', pad=20)plt.savefig('fig7_sample_funnel.png')plt.close()print('Saved fig7_sample_funnel.png')

## Section 2: Score Distribution → Fig 1

In [ ]:
# Figure 1: Literature Rubric score distributionscore_firm = sme[['ticker','normalized_score']].drop_duplicates()fig, axes = plt.subplots(1, 2, figsize=(13, 5))# Histogramaxes[0].hist(score_firm['normalized_score'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)axes[0].axvline(score_firm['normalized_score'].median(), color='red', ls='--', lw=2, label=f'Median={score_firm["normalized_score"].median():.1f}')axes[0].set_xlabel('Literature Rubric Score (1-100)')axes[0].set_ylabel('Number of Firms')axes[0].set_title(f'Score Distribution (N={len(score_firm)})')axes[0].legend()# Box plot by quartilescore_firm['quartile'] = pd.qcut(score_firm['normalized_score'], 4, labels=['Q1\n(Low)', 'Q2', 'Q3', 'Q4\n(High)'])bp = axes[1].boxplot([score_firm[score_firm['quartile']==q]['normalized_score'] for q in score_firm['quartile'].cat.categories],                     labels=score_firm['quartile'].cat.categories, patch_artist=True)colors = ['#d4e6f1', '#85c1e9', '#3498db', '#1a5276']for patch, color in zip(bp['boxes'], colors):    patch.set_facecolor(color)axes[1].set_ylabel('Literature Rubric Score')axes[1].set_title('Score Quartiles')plt.tight_layout()plt.savefig('fig1_score_distribution.png')plt.close()print('Saved fig1_score_distribution.png')print(f'Score stats: mean={score_firm["normalized_score"].mean():.1f}, std={score_firm["normalized_score"].std():.1f}')

## Section 3: Revenue Divergence → Fig 4

In [ ]:
# Figure 4: Revenue divergence by replicability quartilemed_score = sme[['ticker','normalized_score']].drop_duplicates()['normalized_score'].median()sme['high_rep'] = (sme['rho'] >= med_score).astype(int)quarterly = sme.groupby(['period_end','high_rep'])['ln_revenue'].mean().unstack()quarterly.columns = ['Low Replicability', 'High Replicability']# Normalize to pre-shock meanpre_mask = quarterly.index < '2022-10-01'for col in quarterly.columns:    quarterly[col] = quarterly[col] - quarterly.loc[pre_mask, col].mean()fig, ax = plt.subplots(figsize=(12, 6))ax.plot(quarterly.index, quarterly['Low Replicability'], 'b-o', markersize=4, label='Low Replicability (below median)')ax.plot(quarterly.index, quarterly['High Replicability'], 'r-s', markersize=4, label='High Replicability (above median)')ax.axvline(pd.Timestamp('2022-10-01'), color='black', ls='--', lw=2, alpha=0.7)ax.text(pd.Timestamp('2022-11-15'), ax.get_ylim()[1]*0.9, 'ChatGPT\nShock', fontsize=10, ha='left')ax.axhline(0, color='grey', ls=':', alpha=0.5)ax.set_xlabel('Quarter')ax.set_ylabel('Log Revenue (normalized to pre-shock mean)')ax.set_title('Revenue Divergence by LLM Replicability')ax.legend(loc='upper left')plt.xticks(rotation=45)plt.tight_layout()plt.savefig('fig4_revenue_divergence.png')plt.close()print('Saved fig4_revenue_divergence.png')

## Section 4: DiD Results Summary → Table 1

In [ ]:
print('=== TABLE 1: PRIMARY DiD RESULTS ===')print('(From Rscript analysis/did_main.R)')print()t1 = pd.DataFrame({    'Sample':     ['SME <$200M', 'SME <$500M', 'Full B2B SaaS'],    'N':          [116, 146, 168],    'beta':       [-0.798, -0.586, -0.550],    'SE':         [0.369, 0.334, 0.302],    'WCB_p':      [0.020, 0.080, 0.080],    'sig':        ['**', '*', '*'],})print(t1.to_string(index=False))print()print('Placebo (pre-period): beta=-0.276, WCB p=0.353')

## Section 5: Three-Period Analysis → Table 6

In [ ]:
print('=== TABLE 6: THREE-PERIOD ANALYSIS ===')print('Literature Rubric, C1-C10 Continuous Scoring')print()t6 = pd.DataFrame({    'Sample': ['SME <$200M', 'SME <$200M', 'SME <$500M', 'SME <$500M'],    'Period': ['Early AI (2022Q4-2023Q4)', 'Advanced AI (2024Q1+)',                'Early AI (2022Q4-2023Q4)', 'Advanced AI (2024Q1+)'],    'beta': [-0.601, -0.905, -0.424, -0.674],    'WCB_p': [0.213, 0.040, 0.304, 0.091],    'sig': ['', '**', '', '*'],})print(t6.to_string(index=False))print()print('Intensification (SME <$200M): 1.51x (Advanced/Early)')print()print('Quartile DiD: beta=-0.269, WCB p=0.009***')

## Section 6: Score vs Revenue Growth Scatter → Fig 8

In [ ]:
# Figure 8: Score vs post-shock revenue growthgrowth = sme.groupby('ticker').apply(    lambda x: pd.Series({        'delta': x[x['post']==1]['ln_revenue'].mean() - x[x['post']==0]['ln_revenue'].mean(),        'score': x['normalized_score'].iloc[0],        'pre_rev_m': x['pre_rev_m'].iloc[0],    }), include_groups=False).reset_index().dropna()fig, ax = plt.subplots(figsize=(10, 7))scatter = ax.scatter(growth['score'], growth['delta'], c=growth['pre_rev_m'],                      cmap='viridis', s=40, alpha=0.7, edgecolors='grey', lw=0.5)plt.colorbar(scatter, label='Pre-shock Revenue ($M)')# Regression linez = np.polyfit(growth['score'], growth['delta'], 1)p = np.poly1d(z)x_line = np.linspace(growth['score'].min(), growth['score'].max(), 100)ax.plot(x_line, p(x_line), 'r--', lw=2, alpha=0.8)# Label notable firmsfor _, r in growth.iterrows():    if abs(r['delta']) > 1.0 or r['score'] > 65 or r['score'] < 30:        ax.annotate(r['ticker'], (r['score'], r['delta']), fontsize=7, alpha=0.7)corr = growth['score'].corr(growth['delta'])ax.set_xlabel('Literature Rubric Score (higher = more LLM-replicable)')ax.set_ylabel('Post-Shock Revenue Growth (log points)')ax.set_title(f'LLM Replicability vs Revenue Growth (r={corr:.3f}, N={len(growth)})')ax.axhline(0, color='grey', ls=':', alpha=0.5)plt.tight_layout()plt.savefig('fig8_score_revenue_scatter.png')plt.close()print(f'Saved fig8_score_revenue_scatter.png (r={corr:.3f})')

## Section 7: C1-C10 Criterion Analysis

In [ ]:
# Which criteria drive the effect?criteria = ['C1','C2','C3','C4','C5','C6','C7','C8','C9','C10']labels = ['Text Gen', 'Communication', 'Info Synthesis', 'Routine Cognitive',           'Classification', 'Real-Time Data', 'Physical World', 'Prof Judgment',          'Deep Integration', 'Adaptive Response']growth_c = sme.groupby('ticker').apply(    lambda x: pd.Series({        'delta': x[x['post']==1]['ln_revenue'].mean() - x[x['post']==0]['ln_revenue'].mean(),        **{c: x[c].iloc[0] for c in criteria}    }), include_groups=False).reset_index().dropna()corrs = [growth_c[c].corr(growth_c['delta']) for c in criteria]fig, ax = plt.subplots(figsize=(12, 6))colors = ['#e74c3c' if c < 0 else '#27ae60' for c in corrs]bars = ax.barh(range(len(criteria)), corrs, color=colors, edgecolor='white')ax.set_yticks(range(len(criteria)))ax.set_yticklabels([f'{c}: {l}' for c, l in zip(criteria, labels)])ax.set_xlabel('Correlation with Post-Shock Revenue Growth')ax.set_title('Criterion-Level Correlations with Revenue Growth')ax.axvline(0, color='black', lw=0.8)for i, v in enumerate(corrs):    ax.text(v + 0.01 * (1 if v >= 0 else -1), i, f'{v:.3f}', va='center', fontsize=9)plt.tight_layout()plt.savefig('fig10_criterion_correlations.png')plt.close()print('Saved fig10_criterion_correlations.png')print()for c, l, r in zip(criteria, labels, corrs):    print(f'  {c} ({l}): r={r:.3f}')

## Section 8: Final Results Summary

In [ ]:
print('=' * 60)print('FINAL RESULTS SUMMARY')print('Generative AI as a Task Shock — Pipeline v2')print('=' * 60)print()print(f'Universe: {universe.shape[0]} firms (SIC 7370-7379)')print(f'Scored: {scores.shape[0]} firms (C1-C10 continuous rubric)')print(f'B2B SaaS sample: {df["ticker"].nunique()} firms')print(f'Primary sample (SME <$200M): {sme["ticker"].nunique()} firms')print()print('PRIMARY RESULT:')print('  beta = -0.798, WCB p = 0.020**')print('  Interpretation: 1-unit increase in normalized replicability')print('  associated with 80 log-point lower revenue growth post-shock')print()print('THREE-PERIOD:')print('  Early AI (2022Q4-2023Q4): beta = -0.601')print('  Advanced AI (2024Q1+):    beta = -0.905, p = 0.040**')print('  Intensification: 1.51x')print()print('QUARTILE DiD:')print('  Q75 vs Q25: beta = -0.269, WCB p = 0.009***')print()print('PLACEBO: beta = -0.276, p = 0.353 (no pre-trend)')